<!-- NB_DOC_INTRO_V1 -->
# Graph Neural Networks (GNN)

> 📚 **Doc thematique** : [docs/04_DL.md](docs/04_DL.md) (Deep Learning)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**GNN** = Deep Learning sur **graphes** (reseaux sociaux, molecules, recommandations, fraud detection, knowledge graphs). Quand la donnee est naturellement relationnelle, MLP/CNN/RNN perdent la structure d'adjacence — GNN l'exploite via le **message passing** (chaque noeud agrege l'info de ses voisins).

Ce notebook : explication conceptuelle + comparaison architectures (GCN, GraphSAGE, GAT, GIN) + **demo executable from-scratch** d'un GCN simple en numpy (sans torch-geometric, pour la portabilite). Le code PyTorch Geometric (PyG) est fourni en reference.

Versions : `numpy`, `networkx` (visualisation), `torch-geometric` optionnel.

## Plan

1. Pourquoi GNN (vs MLP/CNN/RNN)
2. Concepts cles (nodes, edges, message passing, embedding, pooling)
3. Architectures (GCN, GraphSAGE, GAT, GIN, GraphTransformer, HeteroGNN)
4. Taches sur graphes (node classification, link prediction, graph classification)
5. **Demo from-scratch** : GCN 2-layers en numpy sur Karate Club
6. PyTorch Geometric (PyG) — code de reference
7. Cas d'usage industriels reels
8. Pieges et anti-patterns
9. References


## 1. Pourquoi GNN ?

| Donnee | Representation ideale | Modele |
|---|---|---|
| Image | Grille 2D reguliere | **CNN** (convolutions spatiales) |
| Texte / TS | Sequence 1D | **RNN / Transformer** |
| Tabulaire | Matrice features fixes | **MLP / GBM** |
| **Graphe** | Noeuds + aretes, structure irreguliere | **GNN** |

Un MLP perdrait totalement la structure du graphe. Un CNN a besoin d'une grille fixe. **Un GNN exploite la structure d'adjacence.**

## 2. Concepts cles

| Notion | Sens |
|---|---|
| **Noeud** (vertex) | Entite (user, atome, page web, transaction) |
| **Arete** (edge) | Relation (ami, lien chimique, hyperlien, virement) |
| **Features de noeud** `x_v` | Vecteur attribut du noeud |
| **Features d'arete** `e_uv` | Attribut sur la relation (poids, type, timestamp) |
| **Matrice d'adjacence** `A` | A[i,j] = 1 si arete entre i et j (parfois weighted) |
| **Message Passing** | Chaque noeud agrege l'info de ses voisins (couche par couche) |
| **Embedding de noeud** | Vecteur appris representant chaque noeud |
| **Pooling** | Aggreger les embeddings noeuds → embedding du graphe entier (pour graph classif) |

## 3. Architectures principales

| Modele | Idee | Force |
|---|---|---|
| **GCN** (Kipf & Welling 2017) | Conv = moyenne ponderee normalisee des voisins | Baseline simple |
| **GraphSAGE** (Hamilton 2017) | Sample + aggregate, **inductif** | Scale (gros graphes), generalise a nouveaux noeuds |
| **GAT** (Velickovic 2018) | **Attention** sur les voisins (pondere selon utilite) | Interpretable, robuste |
| **GIN** (Xu 2019) | Theoriquement aussi expressif que WL-test | Classification de graphes |
| **GCN spectral** | Convolution dans le domaine spectral (eigenvectors Laplacian) | Theorie elegante |
| **HeteroGNN** | Graphes multi-types (user / item / etc.) | Vrai monde |
| **GraphTransformer** | Transformer adapte graphes (positional encoding spectral) | Etat de l'art recent (2023+) |

## 4. Taches sur graphes

| Tache | Exemple |
|---|---|
| **Node classification** | Categorie d'un user (fraude / pas fraude), type d'article scientifique |
| **Link prediction** | Suggerer un ami, completer un knowledge graph |
| **Graph classification** | Activite d'une molecule (toxique / pas), categorie d'un programme |
| **Community detection** | Clustering des noeuds (modularite) |
| **Node embedding** | Word2Vec-like pour graphes (DeepWalk, Node2Vec) |


## 5. Demo from-scratch — GCN 2-layers en numpy sur Karate Club

On utilise le **Zachary's Karate Club** : 34 membres d'un club de karate qui s'est scinde en 2 factions. Graphe celebre, ground truth disponible. On va implementer un **GCN** manuellement (sans PyTorch) pour bien comprendre la mecanique.


In [ ]:
# pip install -q numpy networkx scikit-learn matplotlib
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# === Charger le Karate Club graph (built-in networkx) ===
G = nx.karate_club_graph()
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
print(f"Karate Club : {n_nodes} noeuds, {n_edges} aretes")

# Ground truth : le club s'est scinde en 2 factions (Mr. Hi vs Officer)
labels_true = np.array([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in G.nodes()])
print(f"Classes : {np.bincount(labels_true)} (faction 0 / faction 1)")

In [ ]:
# === Matrice d'adjacence A + matrice degre D ===
A = nx.adjacency_matrix(G).toarray().astype(float)
print(f"Adjacency shape : {A.shape}")
print(f"Densite : {A.sum() / (n_nodes**2):.3f}")

# Ajouter self-loops (chaque noeud se voit lui-meme)
A_hat = A + np.eye(n_nodes)

# Matrice degre D^(-1/2) pour normalisation symetrique
D = np.diag(A_hat.sum(axis=1))
D_inv_sqrt = np.linalg.inv(np.sqrt(D))

# Operateur GCN normalise : D^(-1/2) A_hat D^(-1/2)
A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
print(f"A_norm shape : {A_norm.shape}, max value : {A_norm.max():.3f}")

In [ ]:
# === Features initiales : on initialise avec l'identite (chaque noeud = 1-hot) ===
X = np.eye(n_nodes)

# === GCN layer : H' = sigma(A_norm @ H @ W) ===
def gcn_layer(H, W, activation=None):
    Z = A_norm @ H @ W
    if activation == "relu":
        return np.maximum(0, Z)
    elif activation == "tanh":
        return np.tanh(Z)
    return Z

# === Architecture : 2 couches GCN, output dim = 2 (pour visualiser) ===
np.random.seed(SEED)
W1 = np.random.randn(n_nodes, 16) * 0.1
W2 = np.random.randn(16, 2) * 0.1

# Forward pass
H1 = gcn_layer(X, W1, activation="tanh")
H2 = gcn_layer(H1, W2, activation=None)
print(f"Embeddings finaux shape : {H2.shape}")

In [ ]:
# === Visualiser les embeddings 2D + comparer aux vraies factions ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vraies factions sur le graphe
pos = nx.spring_layout(G, seed=SEED)
nx.draw_networkx_nodes(G, pos, node_color=labels_true, cmap=plt.cm.coolwarm,
                       node_size=200, ax=axes[0])
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=axes[0])
nx.draw_networkx_labels(G, pos, font_size=8, ax=axes[0])
axes[0].set_title("Karate Club : vraies factions")
axes[0].axis("off")

# Embeddings GCN (2D), colories par vraie faction
axes[1].scatter(H2[:, 0], H2[:, 1], c=labels_true, cmap=plt.cm.coolwarm, s=80, edgecolors="k")
for i in range(n_nodes):
    axes[1].annotate(str(i), (H2[i, 0], H2[i, 1]), fontsize=7)
axes[1].set_title("Embeddings GCN 2D (couleur = vraie faction)")
axes[1].set(xlabel="dim 0", ylabel="dim 1")
plt.tight_layout()
plt.show()

# Eval : KMeans sur embeddings doit retrouver les factions
km = KMeans(n_clusters=2, random_state=SEED, n_init=10).fit(H2)
acc = max((km.labels_ == labels_true).mean(), (km.labels_ != labels_true).mean())
print(f"\nAccuracy KMeans sur embeddings GCN : {acc:.2%}")
print(f"(50% = random, 100% = parfait)")

**Lecture** : Avec **des poids ALEATOIRES**, un GCN 2-couches separe deja partiellement les 2 factions. Avec un vrai training (loss + backprop), on atteindrait > 95%.

Le **message passing** explique ca : meme sans apprentissage, l'agregation des voisins propage de l'info structurelle (les noeuds proches dans le graphe ont des embeddings proches).


## 6. PyTorch Geometric (PyG) — code de reference

PyG est le standard de facto pour GNN en PyTorch. Installation lourde (~ 1 GB avec torch), donc code de reference :

```python
# pip install -q torch torch_geometric

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid, KarateClub
from torch_geometric.nn import GCNConv, SAGEConv, GATConv

# === Dataset Cora (citation network : 2708 papers, 5429 citations, 7 classes) ===
dataset = Planetoid(root="./data/Cora", name="Cora")
data = dataset[0]
# Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

# === GCN ===
class GCN(torch.nn.Module):
    def __init__(self, n_features, n_classes, hidden=16):
        super().__init__()
        self.conv1 = GCNConv(n_features, hidden)
        self.conv2 = GCNConv(hidden, n_classes)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

model = GCN(dataset.num_node_features, dataset.num_classes)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt.step()

model.eval()
pred = model(data.x, data.edge_index).argmax(dim=1)
test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
print(f"Cora test accuracy : {test_acc:.4f}")  # ~ 0.81 typique
```

### GraphSAGE — inductif, scale
```python
class GraphSAGE(torch.nn.Module):
    def __init__(self, n_features, n_classes, hidden=64):
        super().__init__()
        self.conv1 = SAGEConv(n_features, hidden, aggr="mean")  # 'mean', 'max', 'lstm'
        self.conv2 = SAGEConv(hidden, n_classes, aggr="mean")
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

# Pour gros graphes : NeighborLoader (sample des voisins par batch)
from torch_geometric.loader import NeighborLoader
loader = NeighborLoader(data, num_neighbors=[10, 10], batch_size=128,
                        input_nodes=data.train_mask)
```

### GAT — attention
```python
class GAT(torch.nn.Module):
    def __init__(self, n_features, n_classes, heads=8):
        super().__init__()
        self.gat1 = GATConv(n_features, 8, heads=heads, dropout=0.6)
        self.gat2 = GATConv(8 * heads, n_classes, heads=1, concat=False, dropout=0.6)
    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        return self.gat2(x, edge_index)
```

### Link prediction
```python
from torch_geometric.utils import negative_sampling

class LinkPredictor(torch.nn.Module):
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.encoder = GCN(n_features, hidden)
    def encode(self, x, ei): return self.encoder(x, ei)
    def decode(self, z, ei): return (z[ei[0]] * z[ei[1]]).sum(-1)

# Training : BCE sur edges positifs + negatifs sampled
```


## 7. Cas d'usage industriels reels

| Domaine | Tache | Modele reference |
|---|---|---|
| **Recommendation** (Pinterest) | Link prediction | **PinSage** (GraphSAGE inductif) |
| **Fraud detection** (banque) | Node classification | GCN sur graphe transactions |
| **Drug discovery** | Graph classification | GIN, **MPNN** |
| **Knowledge graphs** | Link prediction | **TransE, ComplEx, RotatE** |
| **Trafic** (Google Maps) | Edge regression | **TGN** (Temporal GNN) |
| **Recommendation user-item** | Bipartite GNN | **LightGCN** |
| **Cybersecurity** | Anomaly detection | DOMINANT |
| **Social network** | Community detection | LINE, Node2Vec |

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Splitter random les noeuds | Leak des aretes vers le test → utiliser `train_mask` PyG |
| Empiler 10 couches GCN | **Over-smoothing** : tous les embeddings convergent |
| GCN transductif sur graphe qui change | GraphSAGE (inductif) — generalise a nouveaux noeuds |
| Ignorer la direction (graphes orientes) | `to_undirected` ou edge type distinct |
| Charger 1 milliard de noeuds en RAM | NeighborSampling / Cluster-GCN |
| Pas de self-loops | A_hat = A + I (sinon le noeud lui-meme s'oublie) |
| Pas de normalisation D^(-1/2) | Explosion des magnitudes |
| Confondre node-level et graph-level | Pooling necessaire pour graph classification |

## 9. References

### Documentation
- **PyTorch Geometric** : https://pytorch-geometric.readthedocs.io/
- **Deep Graph Library (DGL)** : https://www.dgl.ai/  (alternative)
- **Spektral** (Keras) : https://graphneural.network/
- **NetworkX** : https://networkx.org/

### Papers fondateurs
- Kipf & Welling (2017). *Semi-Supervised Classification with Graph Convolutional Networks* (GCN).
- Hamilton et al. (2017). *Inductive Representation Learning on Large Graphs* (GraphSAGE).
- Velickovic et al. (2018). *Graph Attention Networks* (GAT).
- Xu et al. (2019). *How Powerful are Graph Neural Networks?* (GIN).
- Brody et al. (2022). *How Attentive are Graph Attention Networks?* (GATv2).

### Ressources
- Papers with code — GNN : https://paperswithcode.com/area/graphs
- *A Comprehensive Survey on GNNs* (Wu et al., 2021)
- Cours Stanford CS224W (Leskovec) : http://web.stanford.edu/class/cs224w/

Voir aussi :
- [DL_PyTorch.ipynb](./DL_PyTorch.ipynb), [DL_PyTorch_Lightning.ipynb](./DL_PyTorch_Lightning.ipynb)
- [DS_Recommender_Systems.ipynb](./DS_Recommender_Systems.ipynb) — bipartite GNNs (LightGCN)
- [retrieval_BDD_Vectorielle.ipynb](./retrieval_BDD_Vectorielle.ipynb) — embeddings retrieval (cousin de GNN)
